In [ ]:
import csv
import os
from geopy import distance

data_pkg_path = '../data'
input_filename = 'worldcities.csv'
input_path = os.path.join(data_pkg_path, input_filename)
output_filename = 'cities_distance.csv'
output_dir = 'output'
output_path = os.path.join(output_dir, output_filename)

home_city = 'Frankfurt'
home_country = 'Germany'

with open(input_path, 'r', encoding='utf-8') as input_file:
    csv_reader = csv.DictReader(input_file)
    for row in csv_reader:
        if row['city_ascii'] == home_city:
            home_city_coordinates = (row['lat'], row['lng'])
            break

with open(output_path, mode='w', newline='') as output_file:
    fieldnames = ['city', 'distance_from_home']
    csv_writer = csv.DictWriter(output_file, fieldnames=fieldnames)
    csv_writer.writeheader()

    with open(input_path, 'r', encoding='utf-8') as input_file:
        csv_reader = csv.DictReader(input_file)
        for row in csv_reader:
            if (row['country'] == home_country and
                row['city_ascii'] != home_city):
                city_coordinates = (row['lat'], row['lng'])
                city_distance = distance.geodesic(
                    city_coordinates, home_city_coordinates).km
                csv_writer.writerow(
                    {'city': row['city_ascii'],
                     'distance_from_home': city_distance}
                )

print('Successfully written output file at {}'.format(output_path))

Prompt for Claude AI

```
I want to calculate driving distance from home city to all other major cities using OpenRouteService Distance Matrix API. The API supports a maximum of 50 pairs at once. Build a code snippet using Python

<paste the original code using geopy>
```


In [ ]:
import csv
import os
import requests

ORS_API_KEY = 'YOUR_API_KEY'
ORS_URL = 'https://api.openrouteservice.org/v2/matrix/driving-car'

data_pkg_path = 'data'
input_filename = 'worldcities.csv'
input_path = os.path.join(data_pkg_path, input_filename)
output_filename = 'cities_distance_ors.csv'
output_dir = 'output'
output_path = os.path.join(output_dir, output_filename)

if not os.path.exists(output_dir):
    os.mkdir(output_dir)

home_city = 'Bengaluru'
home_country = 'India'

# Find home city coordinates
with open(input_path, 'r', encoding='utf-8') as input_file:
    csv_reader = csv.DictReader(input_file)
    for row in csv_reader:
        if row['city_ascii'] == home_city:
            # ORS expects [lng, lat]
            home_coords = [float(row['lng']), float(row['lat'])]
            break

# Collect all destination cities
destination_cities = []
with open(input_path, 'r', encoding='utf-8') as input_file:
    csv_reader = csv.DictReader(input_file)
    for row in csv_reader:
        if row['country'] == home_country and row['city_ascii'] != home_city:
            destination_cities.append({
                'city': row['city_ascii'],
                'coords': [float(row['lng']), float(row['lat'])]
            })

def get_driving_distances(home_coords, batch_cities):
    """
    Call ORS Distance Matrix API for a batch of destination cities.
    Home city is always index 0 (source), destinations are indices 1..N.
    Max locations per request = 50, so max batch size = 49 destinations.
    """
    locations = [home_coords] + [c['coords'] for c in batch_cities]
    sources = [0]                              # home city as the only source
    destinations = list(range(1, len(locations)))  # all other cities as destinations

    payload = {
        'locations': locations,
        'sources': sources,
        'destinations': destinations,
        'metrics': ['distance'],
        'units': 'km'
    }
    headers = {
        'Authorization': ORS_API_KEY,
        'Content-Type': 'application/json'
    }
    response = requests.post(ORS_URL, json=payload, headers=headers)
    response.raise_for_status()
    # distances[0] is a list of km values from home to each destination
    return response.json()['distances'][0]

# ORS limit: 50 pairs max → with 1 source, max 49 destinations per batch
BATCH_SIZE = 49

with open(output_path, mode='w', newline='') as output_file:
    fieldnames = ['city', 'distance_from_home']
    csv_writer = csv.DictWriter(output_file, fieldnames=fieldnames)
    csv_writer.writeheader()

    for i in range(0, len(destination_cities), BATCH_SIZE):
        batch = destination_cities[i:i + BATCH_SIZE]
        print(f'Processing batch {i // BATCH_SIZE + 1}: cities {i+1}–{i+len(batch)}')

        distances = get_driving_distances(home_coords, batch)

        for city_info, dist in zip(batch, distances):
            csv_writer.writerow({
                'city': city_info['city'],
                # ORS returns None if no driving route exists
                'distance_from_home': round(dist, 2) if dist is not None else 'N/A'
            })

print(f'Successfully written output file at {output_path}')